# Week3_ex1 — Surge impedance loading of a 300 km, 765 kV line

Reproduces Example 5.4 from Glover, Overbye & Sarma, *Power System Analysis and Design* (7th ed.): a lossless 300 km line delivering its surge impedance loading of 2199 MW, then the same line loaded below (1500 MW) and above (2999 MW) that point.

![PowerWorld one-line](Images/Week3_ex1_result_image.png)

In [1]:
# 1. Case data and per-unit conversion
import numpy as np
import pandas as pd
import pandapower as pp

S_BASE = 500.0               # MVA, Simulator Options > Power Flow Solution > General
V_KV = 765.0
F_HZ = 60.0
Z_BASE = V_KV**2 / S_BASE
X_PU, B_PU = 0.08290, 1.66090   # Branches Input, R = 0 (the case treats the line as lossless)
MVA_TOL = 0.5                # PowerWorld MVA convergence tolerance
Z_C = 266.1                  # surge impedance quoted in the textbook, ohm
SIL_MW = V_KV**2 / Z_C

X_OHM = X_PU * Z_BASE
C_NF = B_PU / Z_BASE / (2 * np.pi * F_HZ) * 1e9

pd.DataFrame({
    "Quantity": ["System base", "Line X", "Line B (total)", "Surge impedance loading"],
    "PowerWorld": ["500 MVA", "0.08290 pu", "1.66090 pu", "2199 MW, the load as stored"],
    "pandapower": ["sn_mva=500", f"x_ohm_per_km={X_OHM:.2f}, length_km=1",
                   f"c_nf_per_km={C_NF:.1f}, length_km=1", f"V^2 / Zc = {SIL_MW:.1f} MW"],
})

,Quantity,PowerWorld,pandapower
0,System base,500 MVA,sn_mva=500
1,Line X,0.08290 pu,"x_ohm_per_km=97.03, length_km=1"
2,Line B (total),1.66090 pu,"c_nf_per_km=3764.1, length_km=1"
3,Surge impedance loading,"2199 MW, the load as stored",V^2 / Zc = 2199.3 MW


In [2]:
# 2. Build and solve below, at and above SIL
def build_case(p_mw, q_mvar=0.0):
    net = pp.create_empty_network(f_hz=F_HZ, sn_mva=S_BASE)
    bus_a = pp.create_bus(net, vn_kv=V_KV, name="Bus A")
    bus_b = pp.create_bus(net, vn_kv=V_KV, name="Bus B")
    pp.create_ext_grid(net, bus_a, vm_pu=1.0, name="Slack")
    # pandapower puts half of the line capacitance at each end, which is the same pi model PowerWorld uses
    pp.create_line_from_parameters(net, bus_a, bus_b, length_km=1.0, r_ohm_per_km=0.0, x_ohm_per_km=X_OHM,
                                   c_nf_per_km=C_NF, max_i_ka=10.0, name="300 km line")
    pp.create_load(net, bus_b, p_mw=p_mw, q_mvar=q_mvar, name="Load")
    pp.runpp(net, numba=False)
    return net


def results(net):
    return {
        "Bus B voltage [kV]": net.res_bus.vm_pu.iat[1] * V_KV,
        "Bus B angle [deg]": net.res_bus.va_degree.iat[1],
        "Slack P [MW]": net.res_ext_grid.p_mw.iat[0],
        "Slack Q [Mvar]": net.res_ext_grid.q_mvar.iat[0],
    }


LOADS = [1500.0, 2199.0, 2999.0]
nets = {p: build_case(p) for p in LOADS}

summary = pd.DataFrame({p: results(n) for p, n in nets.items()}).T
summary.index.name = "Load [MW]"
summary.insert(1, "Bus B voltage [pu]", summary["Bus B voltage [kV]"] / V_KV)
summary.round(3)

,Bus B voltage [kV],Bus B voltage [pu],Bus B angle [deg],Slack P [MW],Slack Q [Mvar]
Load [MW],,,,,
1500.0,797.860,1.043,-13.796,1500.0,-492.839
2199.0,765.010,1.000,-21.382,2199.0,-0.154
2999.0,681.823,0.891,-33.910,2999.0,1154.869


In [3]:
# 3. Compare against PowerWorld
# PowerWorld stops iterating once every bus mismatch is below 0.5 MVA, so its answer is only pinned down
# to within that residual. residual_band() pushes +/-0.5 MW and +/-0.5 Mvar into the load and records how far
# each reported quantity moves; a check passes if the error is inside the larger of that band and half the
# last displayed digit.
def residual_band(p_mw):
    base = pd.Series(results(build_case(p_mw)))
    moved = [pd.Series(results(build_case(p_mw + dp, dq))) - base
             for dp, dq in [(MVA_TOL, 0.0), (-MVA_TOL, 0.0), (0.0, MVA_TOL), (0.0, -MVA_TOL)]]
    return pd.concat(moved, axis=1).abs().max(axis=1)


def compare(rows):
    """Build a PowerWorld vs pandapower table and fail loudly on any miss."""
    df = pd.DataFrame(rows, columns=["Case", "Quantity", "PowerWorld", "pandapower", "Tolerance", "Source"])
    df["Abs error"] = (df["pandapower"] - df["PowerWorld"]).abs()
    bad = df[df["Abs error"] > df["Tolerance"]]
    assert bad.empty, bad
    return df.round({"pandapower": 3, "Tolerance": 4, "Abs error": 4})


# PowerWorld values read once from PowerWorld Simulator (Run Mode, Model Explorer > Network > Buses),
# solved again after every load change
PW = {
    1500.0: [797.862, -13.80, 1499.99, -492.86],
    2199.0: [765.005, -21.38, 2199.08, -0.08],
    2999.0: [681.925, -33.90, 2998.75, 1153.73],
}
DISPLAY = [5e-4, 5e-3, MVA_TOL, MVA_TOL]
SOURCE = ["Buses, Volt (kV)", "Buses, Angle (Deg)", "Buses, Gen MW", "Buses, Gen Mvar"]

rows = []
for p, pw_values in PW.items():
    band = residual_band(p)
    pp_values = results(nets[p])
    for (quantity, value), pw, display, source in zip(pp_values.items(), pw_values, DISPLAY, SOURCE):
        rows.append([f"{p:.0f} MW", quantity, pw, value, max(display, band[quantity]), source])

compare(rows)

,Case,Quantity,PowerWorld,pandapower,Tolerance,Source,Abs error
0,1500 MW,Bus B voltage [kV],797.862,797.860,0.0695,"Buses, Volt (kV)",0.0018
1,1500 MW,Bus B angle [deg],-13.800,-13.796,0.0050,"Buses, Angle (Deg)",0.0045
2,1500 MW,Slack P [MW],1499.990,1500.000,0.5000,"Buses, Gen MW",0.0100
3,1500 MW,Slack Q [Mvar],-492.860,-492.839,0.5642,"Buses, Gen Mvar",0.0211
4,2199 MW,Bus B voltage [kV],765.005,765.010,0.0804,"Buses, Volt (kV)",0.0055
5,2199 MW,Bus B angle [deg],-21.380,-21.382,0.0060,"Buses, Angle (Deg)",0.0023
6,2199 MW,Slack P [MW],2199.080,2199.000,0.5000,"Buses, Gen MW",0.0800
7,2199 MW,Slack Q [Mvar],-0.080,-0.154,0.6812,"Buses, Gen Mvar",0.0741
8,2999 MW,Bus B voltage [kV],681.925,681.823,0.1395,"Buses, Volt (kV)",0.1018
9,2999 MW,Bus B angle [deg],-33.900,-33.910,0.0117,"Buses, Angle (Deg)",0.0102


In [4]:
# 4. The 2999 MW point rebuilt from PowerWorld's own residual
# Model Explorer > Mismatches at 2999 MW leaves Bus B at -0.25 MW and -0.20 Mvar (0.32 MVA, inside the 0.5 MVA
# tolerance). PowerWorld's state is then the exact solution for a load of 2999 - 0.25 MW that also supplies
# 0.20 Mvar. Solving that load should land on its numbers to within the precision the residual is shown at.
MISMATCH_MW, MISMATCH_MVAR = -0.25, -0.20
replay = results(build_case(2999.0 + MISMATCH_MW, 0.0 + MISMATCH_MVAR))
plain = results(nets[2999.0])

check = pd.DataFrame({
    "PowerWorld": dict(zip(plain, PW[2999.0])),
    "pandapower, load 2999 MW": plain,
    "pandapower, residual applied": replay,
})
check["Error, plain"] = (check["pandapower, load 2999 MW"] - check["PowerWorld"]).abs()
check["Error, residual applied"] = (check["pandapower, residual applied"] - check["PowerWorld"]).abs()
# The residual itself is shown to 0.01 MW/Mvar, which alone moves the voltage by about 0.002 kV
assert (check["Error, residual applied"] < [3e-3, 5e-3, 0.01, 0.05]).all()
check.round(3)

,PowerWorld,"pandapower, load 2999 MW","pandapower, residual applied","Error, plain","Error, residual applied"
Bus B voltage [kV],681.925,681.823,681.926,0.102,0.001
Bus B angle [deg],-33.900,-33.910,-33.901,0.010,0.001
Slack P [MW],2998.750,2999.000,2998.750,0.250,0.000
Slack Q [Mvar],1153.730,1154.869,1153.727,1.139,0.003


In [5]:
# 5. What SIL means for this line: charging against series reactive loss
# The shunt halves generate B/2 * V^2 at each end; whatever the line absorbs on top of that is I^2 X.
rows = []
for p, net in nets.items():
    v_a, v_b = net.res_bus.vm_pu.iat[0], net.res_bus.vm_pu.iat[1]
    charging = B_PU / 2 * (v_a**2 + v_b**2) * S_BASE
    net_q = net.res_line.ql_mvar.iat[0]
    rows.append({"Load [MW]": p, "Load / SIL [-]": p / SIL_MW, "Charging [Mvar]": charging,
                 "Series I^2 X [Mvar]": net_q + charging, "Net line Q [Mvar]": net_q,
                 "Bus B voltage [pu]": v_b})
balance = pd.DataFrame(rows)

# At SIL the two terms cancel, so the line neither needs nor exports reactive power and the profile is flat
at_sil = balance.set_index("Load [MW]").loc[2199.0]
assert abs(at_sil["Net line Q [Mvar]"]) < 1.0 and abs(at_sil["Bus B voltage [pu]"] - 1.0) < 1e-4
balance.round(3)

,Load [MW],Load / SIL [-],Charging [Mvar],Series I^2 X [Mvar],Net line Q [Mvar],Bus B voltage [pu]
0,1500.0,0.682,866.888,374.049,-492.839,1.043
1,2199.0,1.000,830.461,830.307,-0.154,1.000
2,2999.0,1.364,745.066,1899.935,1154.869,0.891
